# Tourniquet Lot Acceptance Statistical Analysis
## DMDM Lots 1–3 vs. Predicate Devices (NAR, SAM)

Tests performed using the UWO open-source tourniquet tester (Liu et al., HardwareX 2023).  
Goal: determine whether DMDM Lot 3 is non-inferior to predicate devices.

**Calibration note:** Each test session uses an independently calibrated s_value (surface area parameter).  
Raw readings are therefore only directly comparable within the same session.  
Cross-session comparisons use within-session normalization (ratio to predicate peak).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats
from scipy.stats import mannwhitneyu, ttest_1samp, ttest_ind, norm
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.1f}'.format)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Load & Parse Data

In [2]:
TURN_COLS = ['0', '0.25', '0.5', '0.75', '1', '1.25', '1.5', '1.75',
             '2', '2.25', '2.5', '2.75', '3', '3.25', '3.5', '3.75', '4']

df_raw = pd.read_csv(
    r'c:\code\Records\lot_acceptance_testing_data - Sheet1.csv',
    dtype={'Date': str, 'Tourniquet_ID': str, 'Tester': str, 'observations': str}
)
df_raw.columns = df_raw.columns.astype(str).str.strip()

# ------------------------------------------------------------------
# Jan 5 data is duplicated (rows entered twice with a 0.25-col offset).
# Keep only the first occurrence of each (Date, Tourniquet_ID, Tester) group.
df_raw = df_raw.drop_duplicates(subset=['Date', 'Tourniquet_ID', 'Tester'], keep='first').reset_index(drop=True)

# Cast turn columns to numeric
for c in TURN_COLS:
    if c in df_raw.columns:
        df_raw[c] = pd.to_numeric(df_raw[c], errors='coerce')

print(f"{len(df_raw)} unique test records after dedup")
df_raw[['Date','Tourniquet_ID','Tester','s_value','observations']].fillna('').to_string(index=False) |print

32 unique test records after dedup


TypeError: unsupported operand type(s) for |: 'str' and 'builtin_function_or_method'

## 2. Classify Records & Extract Peak Pressure

Each unit is classified as DMDM Lot 1/2/3 or a predicate (NAR / SAM).  
Peak pressure = maximum reading across all turn columns for that test.

In [ ]:
def classify(tid):
    tid = str(tid).upper()
    if 'LOT1' in tid or 'LOT_1' in tid:   return 'DMDM_Lot1'
    if 'LOT2' in tid or 'LOT_2' in tid:   return 'DMDM_Lot2'
    if 'LOT3' in tid or 'LOT_3' in tid:   return 'DMDM_Lot3'
    if 'NAR'  in tid:                      return 'NAR'
    if 'SAM'  in tid:                      return 'SAM'
    if 'GLIA' in tid:                      return 'GLIA'
    if 'RECON' in tid:                     return 'Recon_Medical'
    return 'Other'

df_raw['group']      = df_raw['Tourniquet_ID'].apply(classify)
df_raw['is_used']    = df_raw['Tourniquet_ID'].str.upper().str.startswith('USED')
df_raw['peak']       = df_raw[TURN_COLS].max(axis=1)
df_raw['peak_turn']  = df_raw[TURN_COLS].idxmax(axis=1).astype(float)

# Flag windlass failures (affect pressure retention; distinct from non-critical backplate cracks)
windlass_fail_mask = df_raw['observations'].str.contains('windlass', case=False, na=False)
df_raw['windlass_failure'] = windlass_fail_mask

print(df_raw[['Date','Tourniquet_ID','group','is_used','peak','peak_turn','windlass_failure','observations']]
      .fillna('').to_string(index=False))

## 3. Turn-by-Turn Pressure Curves

Plot all units together per session so the peak-then-slight-decline pattern is visible across all device types.

In [ ]:
COLOR = {
    'DMDM_Lot1': '#2196F3',
    'DMDM_Lot2': '#FF9800',
    'DMDM_Lot3': '#9C27B0',
    'NAR':        '#4CAF50',
    'SAM':        '#F44336',
    'GLIA':       '#795548',
    'Recon_Medical': '#607D8B',
    'Other':      '#9E9E9E',
}
LINESTYLE = {False: '-', True: '--'}   # solid = new, dashed = used

sessions = df_raw['Date'].unique()
fig, axes = plt.subplots(1, len(sessions), figsize=(5.5 * len(sessions), 5), sharey=False)
if len(sessions) == 1:
    axes = [axes]

turn_x = [float(c) for c in TURN_COLS]

for ax, session in zip(axes, sessions):
    sub = df_raw[df_raw['Date'] == session]
    for _, row in sub.iterrows():
        y = [row[c] if c in row.index else np.nan for c in TURN_COLS]
        y = pd.to_numeric(pd.Series(y), errors='coerce').values
        mask = ~np.isnan(y)
        if mask.sum() < 2:
            continue
        color = COLOR.get(row['group'], '#9E9E9E')
        ls    = LINESTYLE[row['is_used']]
        lw    = 2.5 if 'DMDM' in row['group'] else 1.5
        label = row['Tourniquet_ID']
        ax.plot(np.array(turn_x)[mask], y[mask],
                color=color, linestyle=ls, linewidth=lw,
                marker='o', markersize=4, label=label)

    ax.set_title(f"Session: {session}", fontsize=11)
    ax.set_xlabel('Windlass turns')
    ax.set_ylabel('Pressure (raw session units)')
    ax.legend(fontsize=7, loc='upper left')
    ax.xaxis.set_major_locator(ticker.MultipleLocator(0.5))

plt.suptitle('Turn-by-turn pressure — all sessions', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Within-Session Normalization

Because s_value differs per session, absolute values cannot be compared directly across sessions.  
We normalize each DMDM unit's peak by the mean peak of predicate devices tested **in the same session**.  
A ratio ≥ 1.0 means the DMDM unit met or exceeded predicate performance.

In [ ]:
# Mean peak of NEW predicate devices per session (NAR or SAM, not used)
pred_session_mean = (
    df_raw[(df_raw['group'].isin(['NAR', 'SAM'])) & (~df_raw['is_used'])]
    .groupby('Date')['peak']
    .mean()
    .rename('pred_mean_peak')
)
print("Predicate mean peak per session:")
print(pred_session_mean.to_frame())

# Attach normalizer to every DMDM row
dmdm = df_raw[df_raw['group'].isin(['DMDM_Lot1','DMDM_Lot2','DMDM_Lot3'])].copy()
dmdm = dmdm.join(pred_session_mean, on='Date')
dmdm['ratio'] = dmdm['peak'] / dmdm['pred_mean_peak']

print("\nDMDM peak ratios vs. same-session predicate mean:")
print(dmdm[['Date','Tourniquet_ID','group','peak','pred_mean_peak','ratio','windlass_failure','observations']]
      .fillna('').to_string(index=False))

## 4b. Predicate Consistency Check

The NI threshold ratio (0.794) assumes the predicate (NAR) reliably achieves ~315 mmHg across sessions.  
If the predicate varies wildly between sessions, that assumption breaks down and the threshold is unreliable.

**Unit detection heuristic:** Raw values > 100 are assumed to already be in mmHg (s_value applied in firmware).  
Raw values ≤ 100 are assumed to be Newtons — converted here using `P = F / (133.32 × s_value)`.  
This is consistent with the suspected inter-session Arduino firmware configuration difference.

In [ ]:
pred_new = df_raw[
    df_raw['group'].isin(['NAR', 'SAM']) & ~df_raw['is_used']
].copy()

def to_mmhg(row):
    """Values > 100: assume firmware already applied s_value → output is mmHg.
    Values ≤ 100: assume raw Newtons → apply P [mmHg] = F [N] / (133.32 × s_value)."""
    if row['peak'] > 100:
        return row['peak']
    s = row['s_value']
    if pd.notna(s) and float(s) > 0:
        return row['peak'] / (133.32 * float(s))
    return np.nan

pred_new['peak_mmhg'] = pred_new.apply(to_mmhg, axis=1)
pred_new['unit_assumed'] = pred_new['peak'].apply(lambda x: 'mmHg (firmware)' if x > 100 else 'N → converted')

print("New predicate peaks — estimated mmHg:")
print(pred_new[['Date','Tourniquet_ID','s_value','peak','unit_assumed','peak_mmhg']].to_string(index=False))

# ── CV by device type ──────────────────────────────────────────────────────────
print("\nConsistency summary:")
for grp in ['NAR', 'SAM']:
    sub = pred_new[pred_new['group'] == grp]['peak_mmhg'].dropna()
    if len(sub) > 1:
        cv = sub.std(ddof=1) / sub.mean()
        print(f"  {grp}: n={len(sub)}, mean={sub.mean():.1f} mmHg, "
              f"SD={sub.std(ddof=1):.1f}, CV={cv:.1%}")

# ── Focus on sessions that include Lot 3 (the ones that feed the NI test) ─────
lot3_dates = dmdm[dmdm['group'] == 'DMDM_Lot3']['Date'].unique()
pred_lot3  = pred_new[pred_new['Date'].isin(lot3_dates)]

print(f"\nPredicate measurements in Lot 3 sessions ({list(lot3_dates)}):")
print(pred_lot3[['Date','Tourniquet_ID','peak_mmhg']].to_string(index=False))

nar_lot3 = pred_lot3[pred_lot3['group'] == 'NAR']['peak_mmhg'].dropna()
ni_ref   = nar_lot3.mean()
ni_delta = (ni_ref - 250) / ni_ref
ni_thresh = 250 / ni_ref

print(f"\n── NI reference (mean NAR in Lot 3 sessions): {ni_ref:.1f} mmHg  (n={len(nar_lot3)})")
print(f"   δ  = ({ni_ref:.1f} − 250) / {ni_ref:.1f}  = {ni_delta:.3f}  ({ni_delta:.1%})")
print(f"   NI threshold ratio  = 250 / {ni_ref:.1f}  = {ni_thresh:.3f}")
if len(nar_lot3) > 1:
    nar_cv = nar_lot3.std(ddof=1) / nar_lot3.mean()
    print(f"   NAR CV in Lot 3 sessions: {nar_cv:.1%}  — ", end='')
    print("✓ consistent (< 5%)" if nar_cv < 0.05 else
          "acceptable (< 15%)" if nar_cv < 0.15 else
          "⚠ high variability — constancy assumption is weak")

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: all sessions
ax = axes[0]
CPRED = {'NAR': '#4CAF50', 'SAM': '#F44336'}
MPRED = {'NAR': 'o',       'SAM': 's'}

# sort by date for a clean x-axis
pred_new_sorted = pred_new.sort_values('Date')
dates_all = pred_new_sorted['Date'].unique()
date_idx  = {d: i for i, d in enumerate(dates_all)}

for grp, gdf in pred_new_sorted.groupby('group'):
    xs = [date_idx[d] + (0.1 if grp == 'SAM' else -0.1) for d in gdf['Date']]
    ax.scatter(xs, gdf['peak_mmhg'],
               color=CPRED[grp], marker=MPRED[grp], s=80, zorder=3, label=grp)
    ax.axhline(gdf['peak_mmhg'].mean(),
               color=CPRED[grp], linestyle='--', lw=1, alpha=0.5,
               label=f'{grp} mean = {gdf["peak_mmhg"].mean():.0f} mmHg')

ax.axhline(250, color='black', lw=1.5, linestyle=':', label='Clinical floor 250 mmHg')
ax.set_xticks(range(len(dates_all)))
ax.set_xticklabels(dates_all, rotation=20, ha='right')
ax.set_ylabel('Estimated peak (mmHg)')
ax.set_title('New predicate peaks — all sessions\n(dashed = group mean)')
ax.legend(fontsize=8)

# Right: Lot 3 sessions only, annotated with the NI threshold
ax2 = axes[1]
lot3_dates_sorted = sorted(lot3_dates)
date_idx2 = {d: i for i, d in enumerate(lot3_dates_sorted)}

for grp, gdf in pred_lot3.groupby('group'):
    xs = [date_idx2[d] + (0.1 if grp == 'SAM' else -0.1) for d in gdf['Date']]
    ax2.scatter(xs, gdf['peak_mmhg'],
                color=CPRED[grp], marker=MPRED[grp], s=100, zorder=3, label=grp)
    for x, (_, row) in zip(xs, gdf.iterrows()):
        ax2.annotate(f"{row['peak_mmhg']:.0f}", (x, row['peak_mmhg']),
                     textcoords='offset points', xytext=(0, 8), fontsize=8, ha='center')

ax2.axhline(ni_ref,   color='grey',  lw=1.5, linestyle='--',
            label=f'NAR mean ref = {ni_ref:.0f} mmHg')
ax2.axhline(ni_thresh * ni_ref, color='purple', lw=1.5, linestyle='-.',
            label=f'NI threshold = {ni_thresh*ni_ref:.0f} mmHg ({ni_thresh:.3f} × ref)')
ax2.axhline(250, color='black', lw=1.5, linestyle=':', label='Clinical floor 250 mmHg')
ax2.set_xticks(range(len(lot3_dates_sorted)))
ax2.set_xticklabels(lot3_dates_sorted, rotation=20, ha='right')
ax2.set_ylabel('Estimated peak (mmHg)')
ax2.set_title('Predicate peaks — Lot 3 sessions only\n(these sessions set the NI reference)')
ax2.legend(fontsize=8)

plt.suptitle('Predicate Consistency Check', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Descriptive Statistics by Lot

In [ ]:
def describe_group(df, label):
    r = df['ratio'].dropna()
    ci_lo, ci_hi = stats.t.interval(0.95, df=len(r)-1, loc=r.mean(), scale=stats.sem(r))
    return {
        'Group': label,
        'n': len(r),
        'Mean ratio': r.mean(),
        'SD': r.std(ddof=1),
        'Median ratio': r.median(),
        'Min': r.min(),
        'Max': r.max(),
        '95% CI low': ci_lo,
        '95% CI high': ci_hi,
    }

rows = []
for lot in ['DMDM_Lot1', 'DMDM_Lot2', 'DMDM_Lot3']:
    sub = dmdm[dmdm['group'] == lot]
    rows.append(describe_group(sub, lot))
    rows.append(describe_group(sub[~sub['windlass_failure']], f'{lot} (no windlass fail)'))

desc = pd.DataFrame(rows).set_index('Group')
desc

## 6. Non-Inferiority Test

**H₀:** Mean Lot 3 ratio ≤ (1 − δ)  — i.e., Lot 3 is inferior by more than margin δ  
**H₁:** Mean Lot 3 ratio > (1 − δ)  — i.e., Lot 3 is non-inferior  

We test a range of δ values (5 %, 10 %, 15 %, 20 %) and report the one-sided p-value.  
A pre-specified δ should be chosen based on clinical judgement *before* looking at the data.

### What the non-inferiority test is actually doing

**The core question:** Does the DMDM tourniquet generate enough pressure to work, even if it doesn't match the predicate exactly?

**Step 1 — Set the clinical floor at 250 mmHg.**  
This is the minimum pressure we believe reliably occludes most limbs. It is conservative: surgical literature shows effective occlusion averaging 140–230 mmHg depending on limb, but we use 250 as an accepted safety margin. (Note: this floor is borrowed from pneumatic surgical literature — no validated floor exists for field windlass tourniquets.)

**Step 2 — Find the acceptable loss.**  
The predicate (NAR) achieves roughly 315 mmHg in the same sessions as Lot 3.  
The gap between predicate and floor: **315 − 250 = 65 mmHg** — this is the performance "room" the predicate has above the minimum.

**Step 3 — Express that room as a fraction (δ, the NI margin).**  
δ = 65 / 315 = **20.6%**  
Our device is allowed to fall up to 20.6% below the predicate, because even at that deficit it still reaches 250 mmHg.

**Step 4 — Convert to a ratio threshold.**  
NI threshold = 1 − δ = **0.794** → the device must achieve at least 79.4% of what the predicate achieves in the same session.

**Why use a ratio instead of comparing raw values to 250 mmHg directly?**  
Different sessions have different calibrations and possibly different output units (see Section 4b). The ratio DMDM_peak / predicate_peak in the *same* session cancels out the calibration factor, giving a comparison that is valid regardless of units.

**The test in one line:**
> For each DMDM unit: `DMDM_peak / same-session predicate_peak ≥ 0.794` → **PASS**

Note: the threshold 0.794 = 250 / 314 is computed from the predicate consistency check above and held fixed. It is not re-fitted per session.

In [ ]:
lot3_all   = dmdm[dmdm['group'] == 'DMDM_Lot3']['ratio'].dropna()
lot3_clean = dmdm[(dmdm['group'] == 'DMDM_Lot3') & (~dmdm['windlass_failure'])]['ratio'].dropna()

print(f"Lot 3 all units          (n={len(lot3_all)}): ratios = {lot3_all.round(3).tolist()}")
print(f"Lot 3 no windlass fail   (n={len(lot3_clean)}): ratios = {lot3_clean.round(3).tolist()}")
print(f"\nPre-specified NI threshold (from Section 4b): {ni_thresh:.3f}  (δ = {ni_delta:.1%})")
print()

ni_results = []

# ── Primary test: pre-specified threshold derived from predicate consistency check ──
for label, series in [('All units', lot3_all), ('No windlass fail', lot3_clean)]:
    t_stat, p_two = ttest_1samp(series, popmean=ni_thresh)
    p_one = p_two / 2 if t_stat > 0 else 1 - p_two / 2
    ni_results.append({
        'δ': f'{ni_delta:.1%}  ← pre-specified',
        'Subset': label,
        'n': len(series),
        'Mean ratio': round(series.mean(), 3),
        'NI threshold': round(ni_thresh, 3),
        't-stat': round(t_stat, 3),
        'p (one-sided)': round(p_one, 4),
        'Non-inferior α=0.05?': '✓ YES' if p_one < 0.05 else '✗ NO',
    })

# ── Sensitivity sweep across other margins for reference ──────────────────────
for delta in [0.05, 0.10, 0.15, 0.20, 0.25]:
    threshold = 1.0 - delta
    for label, series in [('All units', lot3_all), ('No windlass fail', lot3_clean)]:
        t_stat, p_two = ttest_1samp(series, popmean=threshold)
        p_one = p_two / 2 if t_stat > 0 else 1 - p_two / 2
        ni_results.append({
            'δ': f'{delta:.0%}',
            'Subset': label,
            'n': len(series),
            'Mean ratio': round(series.mean(), 3),
            'NI threshold': threshold,
            't-stat': round(t_stat, 3),
            'p (one-sided)': round(p_one, 4),
            'Non-inferior α=0.05?': '✓ YES' if p_one < 0.05 else '✗ NO',
        })

ni_df = pd.DataFrame(ni_results)
ni_df

## 7. Mann-Whitney U Test — Lot 3 vs. Lots 1 & 2

Non-parametric test — no normality assumption. Tests whether Lot 3 peak ratios  
are drawn from the same distribution as Lot 1 / Lot 2.

In [ ]:
lot1 = dmdm[dmdm['group'] == 'DMDM_Lot1']['ratio'].dropna()
lot2 = dmdm[dmdm['group'] == 'DMDM_Lot2']['ratio'].dropna()
lot3 = lot3_all.copy()

mw_results = []
for a_label, a_vals in [('Lot1', lot1), ('Lot2', lot2), ('Lot1+Lot2', pd.concat([lot1, lot2]))]:
    u, p = mannwhitneyu(a_vals, lot3, alternative='two-sided')
    mw_results.append({
        'Comparison': f'{a_label} vs Lot3',
        f'n ({a_label})': len(a_vals),
        'n (Lot3)': len(lot3),
        'U statistic': u,
        'p (two-sided)': round(p, 4),
        'Significant at α=0.05?': '✓ YES' if p < 0.05 else '✗ NO',
    })

pd.DataFrame(mw_results)

## 8. Bootstrap Confidence Intervals on Mean Ratio

With small n, bootstrap CIs are more reliable than t-based CIs.  
10,000 resamples with replacement.

In [ ]:
rng = np.random.default_rng(42)

def bootstrap_ci(data, n_boot=10_000, ci=0.95):
    data = np.asarray(data)
    boot_means = [rng.choice(data, size=len(data), replace=True).mean() for _ in range(n_boot)]
    lo = np.percentile(boot_means, (1 - ci) / 2 * 100)
    hi = np.percentile(boot_means, (1 + ci) / 2 * 100)
    return lo, hi

boot_rows = []
for lot, series in [('Lot1', lot1), ('Lot2', lot2), ('Lot3 (all)', lot3_all), ('Lot3 (no windlass fail)', lot3_clean)]:
    lo, hi = bootstrap_ci(series)
    boot_rows.append({
        'Group': lot,
        'n': len(series),
        'Mean ratio': series.mean().round(3),
        'Bootstrap 95% CI': f'[{lo:.3f}, {hi:.3f}]',
        'CI includes 1.0?': '✓' if lo <= 1.0 <= hi else '✗',
        'CI lower bound ≥ 0.80?': '✓' if lo >= 0.80 else '✗',
    })

pd.DataFrame(boot_rows).set_index('Group')

## 9. Lot 3 Outlier: lot3_9

This unit had near-zero pressure even after calibration correction and is a clear outlier.  
We test whether removing it changes the conclusions, and flag it for investigation.

In [ ]:
lot3_no_outlier = dmdm[
    (dmdm['group'] == 'DMDM_Lot3') &
    (~dmdm['Tourniquet_ID'].str.lower().str.contains('lot3_9'))
]['ratio'].dropna()

# Z-score of lot3_9
lot3_9_ratio = dmdm[dmdm['Tourniquet_ID'].str.lower().str.contains('lot3_9')]['ratio']
z = (lot3_9_ratio.values[0] - lot3_all.mean()) / lot3_all.std() if len(lot3_9_ratio) else None

print(f"lot3_9 ratio: {lot3_9_ratio.values}")
print(f"Lot3 mean ratio: {lot3_all.mean():.3f}  SD: {lot3_all.std():.3f}")
print(f"lot3_9 z-score: {z:.2f}" if z else "lot3_9 not found in DMDM subset")
print(f"\nLot3 mean ratio WITHOUT lot3_9: {lot3_no_outlier.mean():.3f}")
print(f"Lot3 mean ratio WITH    lot3_9: {lot3_all.mean():.3f}")

## 10. Summary Visualization — Ratio Distribution by Lot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: strip / dot plot of individual ratios ---
ax = axes[0]
lots = ['DMDM_Lot1', 'DMDM_Lot2', 'DMDM_Lot3']
lot_labels = ['Lot 1', 'Lot 2', 'Lot 3']
x_pos = {l: i for i, l in enumerate(lots)}

for lot in lots:
    sub = dmdm[dmdm['group'] == lot]
    xs = [x_pos[lot]] * len(sub)
    # jitter
    xs = np.array(xs) + rng.uniform(-0.08, 0.08, size=len(sub))
    colors = ['red' if wf else COLOR[lot] for wf in sub['windlass_failure']]
    ax.scatter(xs, sub['ratio'], c=colors, s=60, zorder=3, edgecolors='white', linewidths=0.5)
    # mean ± 95% CI
    lo, hi = bootstrap_ci(sub['ratio'].dropna())
    ax.plot([x_pos[lot]-0.18, x_pos[lot]+0.18], [sub['ratio'].mean()]*2, color='black', lw=2, zorder=4)
    ax.errorbar(x_pos[lot], sub['ratio'].mean(), yerr=[[sub['ratio'].mean()-lo], [hi-sub['ratio'].mean()]],
                fmt='none', color='black', capsize=6, lw=1.5, zorder=4)

for delta, ls in [(0.05,'--'), (0.10,'-.'), (0.15,':'), (0.20,(0,(3,5,1,5)))]:
    ax.axhline(1-delta, color='grey', linestyle=ls, lw=1,
               label=f'NI threshold δ={delta:.0%} ({1-delta:.2f})')
ax.axhline(1.0, color='black', lw=0.8, alpha=0.4)

ax.set_xticks(range(len(lots)))
ax.set_xticklabels(lot_labels)
ax.set_ylabel('Peak pressure ratio vs. same-session predicate mean')
ax.set_title('Individual unit ratios\n(red = windlass failure, bar = mean ± 95% CI)')
ax.legend(fontsize=8, loc='lower right')
ax.set_ylim(bottom=0)

# --- Right: box plot of ratios ---
ax2 = axes[1]
data_to_plot = [dmdm[dmdm['group'] == l]['ratio'].dropna().values for l in lots]
bp = ax2.boxplot(data_to_plot, labels=lot_labels, patch_artist=True,
                 medianprops=dict(color='black', lw=2))
for patch, lot in zip(bp['boxes'], lots):
    patch.set_facecolor(COLOR[lot])
    patch.set_alpha(0.7)

for delta, ls in [(0.05,'--'), (0.10,'-.'), (0.15,':')]:
    ax2.axhline(1-delta, color='grey', linestyle=ls, lw=1, label=f'δ={delta:.0%}')
ax2.axhline(1.0, color='black', lw=0.8, alpha=0.4)
ax2.set_ylabel('Peak pressure ratio vs. same-session predicate mean')
ax2.set_title('Distribution by lot (box = IQR, whiskers = range)')
ax2.legend(fontsize=8)
ax2.set_ylim(bottom=0)

plt.suptitle('DMDM Performance Relative to Same-Session Predicate (NAR/SAM)', fontsize=12)
plt.tight_layout()
plt.show()

## 11. Interpretation Notes

- **Non-inferiority margin δ must be pre-specified** based on clinical judgement before analyzing data.  
  A common starting point for life-critical devices is δ = 10–15%.  
- **lot3_9** is a statistical outlier and should be investigated for assembly defect before being included  
  in final acceptance conclusions. Its inclusion materially lowers the Lot 3 mean ratio.  
- **Windlass failures** (Lot3_2 and Lot3_7) are mechanically distinct from non-critical backplate cracks:  
  a failed windlass may not maintain twist and therefore may not hold pressure.  
  Recommend analysing with and without these units and reporting both.  
- **Sample sizes are small** (n = 5–9 per lot). All p-values should be interpreted with caution.  
  These results are appropriate for internal lot release decisions and CAPA support,  
  but a larger study would be needed for regulatory non-inferiority claims.  
- The Jan 5 duplicate rows have been removed; only one record per (Date, Tourniquet_ID, Tester) is used.